# Week 8 — Stack Health Monitor (SamuelAdebodun)

Scans a repo for **requirements.txt**, **package.json**, and **Dockerfile** (FROM), enriches with latest-version info, and gets an LLM report: **Upgrade** / **Replace** / **Watch**. Uses OpenAI, Anthropic, or Ollama. Optional RAG context and memory.json for "what changed since last run."

In [8]:
import os
import re
import json
from pathlib import Path
from typing import Optional, List
from dotenv import load_dotenv
from openai import OpenAI
import anthropic
import gradio as gr

In [9]:
load_dotenv(override=True)
openai_key = os.getenv("OPENAI_API_KEY")
anthropic_key = os.getenv("ANTHROPIC_API_KEY")

MODEL_OPENAI = "gpt-4o-mini"
MODEL_ANTHROPIC = "claude-3-5-sonnet-20241022"
MODEL_OLLAMA = "llama3.2"

openai_client = OpenAI() if openai_key else None
anthropic_client = anthropic.Anthropic() if anthropic_key else None
ollama_client = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
print("Clients ready.")

Clients ready.


In [10]:
def _find_requirements(path: Path) -> Optional[Path]:
    path = path.resolve()
    if not path.exists():
        return None
    if path.is_file() and path.name == "requirements.txt":
        return path
    if path.is_dir():
        candidate = path / "requirements.txt"
        if candidate.exists():
            return candidate
        for child in path.iterdir():
            if child.is_file() and child.name == "requirements.txt":
                return child
            if child.is_dir():
                c2 = child / "requirements.txt"
                if c2.exists():
                    return c2
    return None

def _parse_requirements(req_file: Path) -> list[dict]:
    deps = []
    for line in req_file.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.split("#")[0].strip()
        if not line or line.startswith("-"):
            continue
        m = re.match(r"([a-zA-Z0-9_.-]+)\s*([=<>!~]+)\s*(.+)", line)
        if m:
            name, ver = m.group(1), m.group(3).strip().split()[0] if m.group(3) else "?"
            deps.append({"name": name, "version": ver, "source": "requirements.txt"})
        else:
            name = re.match(r"^([a-zA-Z0-9_.-]+)", line)
            if name:
                deps.append({"name": name.group(1), "version": "?", "source": "requirements.txt"})
    return deps

def scan_requirements(repo_path: str) -> list[dict]:
    path = Path(repo_path.strip())
    req_file = _find_requirements(path)
    if not req_file:
        return []
    return _parse_requirements(req_file)

def scan_package_json(repo_path: str) -> list[dict]:
    path = Path(repo_path.strip()).resolve()
    if path.is_file():
        path = path.parent
    if not path.exists():
        return []
    pkg = path / "package.json"
    if not pkg.exists():
        for child in path.iterdir():
            if child.is_dir() and (child / "package.json").exists():
                pkg = child / "package.json"
                break
    if not pkg.exists():
        return []
    try:
        import json
        data = json.loads(pkg.read_text(encoding="utf-8", errors="ignore"))
        deps = []
        for key in ("dependencies", "devDependencies"):
            for name, ver in (data.get(key) or {}).items():
                ver = ver.strip('"^~') if isinstance(ver, str) else "?"
                deps.append({"name": name, "version": ver, "source": "package.json"})
        return deps
    except Exception:
        return []

def scan_dockerfile(repo_path: str) -> list[dict]:
    path = Path(repo_path.strip()).resolve()
    if path.is_file():
        path = path.parent
    if not path.exists():
        return []
    found = []
    for f in path.rglob("Dockerfile*"):
        if not f.is_file():
            continue
        text = f.read_text(encoding="utf-8", errors="ignore")
        for line in text.splitlines():
            if line.strip().upper().startswith("FROM "):
                parts = line.strip().split(None, 1)
                image = parts[1].split("#")[0].strip() if len(parts) > 1 else "?"
                if image and image != "scratch":
                    found.append({"name": image, "version": "?", "source": f.name})
    return found[:20]

def scan_all(repo_path: str) -> list[dict]:
    raw = repo_path.strip()
    if raw.lower().startswith(("http://", "https://")):
        return []
    path = Path(raw).resolve()
    if not path.exists():
        return []
    deps = scan_requirements(repo_path) + scan_package_json(repo_path) + scan_dockerfile(repo_path)
    return deps

In [11]:
MOCK_LATEST = {
    "requests": "2.31.0", "flask": "3.0.0", "django": "5.0", "numpy": "2.0.0",
    "pandas": "2.2.0", "openai": "1.12.0", "anthropic": "0.18.0", "gradio": "4.44.0",
    "python-dotenv": "1.0.0", "chromadb": "0.4.22", "transformers": "4.45.0",
    "langchain": "0.3.0", "langchain-core": "0.3.0", "langchain-openai": "0.2.0",
    "pydantic": "2.9.0", "torch": "2.5.0", "datasets": "3.2.0", "plotly": "5.24.0",
    "beautifulsoup4": "4.12.0", "sentence-transformers": "3.2.0", "modal": "0.64.0",
    "ipykernel": "6.29.0", "tqdm": "4.66.0", "scikit-learn": "1.5.0", "matplotlib": "3.9.0",
}

def _norm(name: str) -> str:
    return name.lower().replace("_", "-")

def enrich(deps):
    for d in deps:
        key = _norm(d["name"]) if d.get("source") in ("requirements.txt", "package.json") else d["name"]
        latest = MOCK_LATEST.get(key, "?")
        d["latest"] = latest
        d["status"] = "ok" if latest == "?" or str(d["version"]).strip() == latest else "outdated"
    return deps

In [12]:
# Simple RAG: short upgrade notes (no vector DB for simplicity)
RAG_NOTES = [
    "Django 4.x to 5.0: check deprecation of django.utils.timezone.utc; run tests.",
    "Node 18 LTS is supported until Apr 2025; Node 20 LTS recommended for new projects.",
    "Python 3.12: ensure all deps have 3.12 wheels; numpy/pandas are fine.",
    "Docker Alpine 3.18+: use apk --no-cache for smaller images.",
]

MEMORY_FILE = "memory.json"

def load_memory():
    if os.path.exists(MEMORY_FILE):
        try:
            with open(MEMORY_FILE) as f:
                return json.load(f)
        except Exception:
            pass
    return {"names": []}

def save_memory(deps):
    names = [d["name"] for d in deps]
    with open(MEMORY_FILE, "w") as f:
        json.dump({"names": names}, f, indent=2)

In [13]:
def run_planner(deps, model_choice: str, rag_notes: Optional[List[str]] = None) -> str:
    if not deps:
        return "No dependencies found. Use a path to a folder that contains requirements.txt, package.json, or a Dockerfile."
    lines = [f"- {d['name']} {d['version']} (latest: {d['latest']}, {d['status']}) [{d.get('source', '?')}]" for d in deps]
    context = ""
    if rag_notes:
        context = "\nRelevant upgrade notes to consider:\n" + "\n".join(f"- {n}" for n in rag_notes) + "\n\n"
    prompt = (
        "Given these dependencies, write a short report. Use three sections: **Upgrade** (safe to upgrade), **Replace** (deprecated or swap), **Watch** (security or aging). One line per item with a brief reason. Be concise.\n\n"
        + context
        + "Dependencies:\n" + "\n".join(lines)
    )
    try:
        if model_choice == "OpenAI (gpt-4o-mini)":
            if not openai_client:
                return "OpenAI client not set."
            r = openai_client.chat.completions.create(model=MODEL_OPENAI, messages=[{"role": "user", "content": prompt}], temperature=0)
            return r.choices[0].message.content or ""
        elif model_choice == "Anthropic (claude-3-5-sonnet)":
            if not anthropic_client:
                return "Anthropic client not set."
            m = anthropic_client.messages.create(model=MODEL_ANTHROPIC, max_tokens=400, messages=[{"role": "user", "content": prompt}])
            return m.content[0].text if m.content else ""
        else:
            r = ollama_client.chat.completions.create(model=MODEL_OLLAMA, messages=[{"role": "user", "content": prompt}], temperature=0)
            return r.choices[0].message.content or ""
    except Exception as e:
        return f"Error: {e}"

In [14]:
def run_report(repo_path: str, model_choice: str) -> str:
    raw = repo_path.strip()
    if raw.lower().startswith(("http://", "https://")):
        return "Please use a **local folder path** (e.g. /path/to/repo or .), not a URL. Clone the repo first, then enter that folder path."
    deps = scan_all(repo_path)
    deps = enrich(deps)
    prev = load_memory()
    prev_names = set(prev.get("names", []))
    curr_names = {d["name"] for d in deps}
    new_deps = curr_names - prev_names
    removed = prev_names - curr_names
    save_memory(deps)
    report = run_planner(deps, model_choice, RAG_NOTES)
    if new_deps or removed:
        head = "**Since last run:** "
        if new_deps:
            head += f"New: {', '.join(sorted(new_deps)[:10])}{'…' if len(new_deps) > 10 else ''}. "
        if removed:
            head += f"Removed: {', '.join(sorted(removed)[:10])}{'…' if len(removed) > 10 else ''}.\n\n"
        report = head + report
    return report

In [ ]:
with gr.Blocks(title="Stack Health") as demo:
    gr.Markdown("## Stack Health Monitor — requirements.txt, package.json, Dockerfile")
    path_in = gr.Textbox(label="Path to repo or folder", placeholder="/path/to/repo or . for current dir")
    model_dd = gr.Dropdown(choices=["OpenAI (gpt-4o-mini)", "Anthropic (claude-3-5-sonnet)", "Ollama (llama3.2)"], value="OpenAI (gpt-4o-mini)", label="Model")
    btn = gr.Button("Run report")
    out = gr.Textbox(label="Report (Upgrade / Replace / Watch)", lines=12)
    btn.click(fn=run_report, inputs=[path_in, model_dd], outputs=out)
demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/root/Projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/root/Projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/root/Projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/root/Projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1623, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/root/Projects/llm_engineering/.venv/lib/python3.12/site-p